In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 11,          # mismo tamaño que \documentclass[11pt]{article}
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "text.latex.preamble": r"\usepackage{amsmath}",
})

In [ ]:
from cycler import cycler

plt.rcParams["axes.prop_cycle"] = cycler(color=[
    "#0072B2", "#E69F00", "#009E73", "#D55E00",
    "#CC79A7", "#56B4E9", "#F0E442", "#000000"
])

# Fiat-Shamir: de protocolo interactivo a firma no interactiva

**Trabajo Final — Programación Científica 2026-1**

Integrantes:\
Delgado Ortiz, David \
Fonseca Aldana, Miguel Angel\
Moreno Ceballos, Jose Daniel\
Ospina Ocampo, Juan Diego\
Urrutia Manyoma, Haison



##  Contexto teórico

El protocolo de **Fiat-Shamir** permite a un **Prover** convencer a un **Verifier** de que conoce un secreto \(s\) sin revelarlo.

Originalmente, es un protocolo **interactivo** que consta de tres pasos:

1. **Compromiso (Commitment):**

   $$
   x = r^2 \pmod{N}
   $$

2. **Reto (Challenge):**

   $$
   e \in \{0,1\}
   $$

3. **Respuesta (Response):**

   $$
   y = r \cdot s^e \pmod{N}
   $$

### La transformación de Fiat-Shamir (1986)

En lugar de que el **Verifier** elija el reto, éste se calcula como

$$
e = H(x),
$$

donde \(H\) es una función hash criptográfica.

Esta transformación elimina la necesidad de interacción entre el **Prover** y el **Verifier**, convirtiendo el protocolo de identificación en un **esquema de firma digital**.


### ¿Por qué es seguro el protocolo? El rol del reto $e \in \{0,1\}$

Sea $N = p \cdot q$ el producto de dos primos, y $v = s^2 \bmod N$ el valor público asociado
al secreto $s$. La seguridad de Fiat-Shamir descansa en un hecho estructural: **si alguien
logra encontrar dos raíces cuadradas distintas y no triviales de $v$ módulo $N$, puede
factorizar $N$**, lo cual es computacionalmente inviable (NP-completo) para $N$ suficientemente grande.


#### El protocolo, ronda por ronda

1. **Compromiso:** Alice elige $r$ aleatorio (nuevo en cada ronda) y calcula
   $$x = r^2 \bmod N$$

2. **Reto:** Bob (o el hash, en la versión no interactiva) elige $e \in \{0,1\}$

3. **Respuesta:**
   - si $e=0$: Alice manda $y = r$, y Bob verifica $y^2 \equiv x \pmod N$
   - si $e=1$: Alice manda $y = r \cdot s \bmod N$, y Bob verifica $y^2 \equiv x \cdot v \pmod N$

Nótese que solo la rama $e=1$ involucra el secreto $s$. La rama $e=0$ únicamente confirma
que Alice conocía el $r$ usado al comprometerse.

#### Por qué un impostor no puede responder a ambos retos

Un impostor que **no conoce $s$** puede intentar hacer trampa invirtiendo el orden de
construcción: en vez de generar $x=r^2$ primero, elige un $y$ arbitrario y despeja

$$x \equiv y^2 \cdot v^{-1} \pmod N$$

Con este $x$ "preparado", el impostor **sí** puede responder correctamente si Bob pide
$e=1$ (manda el $y$ que ya tenía). Pero si Bob pide $e=0$, necesitaría una raíz cuadrada
de ese mismo $x$ — y encontrarla es tan difícil como el problema original de raíces
cuadradas módulo $N$, del que depende toda la seguridad del esquema.

#### Probabilidad de éxito de un impostor

Como el impostor debe adivinar de antemano cuál reto le tocará, su probabilidad de éxito en una
ronda es $1/2$, y cae a $2^{-k}$ al repetir el protocolo $k$ veces de forma independiente.

## Implementación del protocolo Fiat-Shamir

### Configuración: generación de N = p·q y del secreto

En esta etapa se generan los parámetros globales del protocolo mediante los siguientes pasos:
- Módulo del sistema ($N$): Se eligen dos números primos grandes $p$ y $q$ de forma independiente y aleatoria. Se calcula $N = p \cdot q$. El valor de $N$ se hace público, mientras que $p$ y $q$ permanecen secretos.
- Clave privada ($s$): El Prover elige un número entero secreto $s$ en el rango $1 < s < N$ tal que $\gcd(s, N) = 1$ (es decir, $s$ es coprimo con $N$).
- Clave pública ($v$): El Prover calcula su clave pública mediante la congruencia:$$v \equiv s^2 \pmod N$$
- Oráculo Aleatorio (Heurística de Fiat-Shamir): Para eliminar la interacción directa con el Verifier, se utiliza una función hash criptográfica $H(\cdot)$ (como SHA-256) que actúa como un oráculo aleatorio para generar el reto $e \in \{0, 1\}$ a partir del compromiso $$ x:e = H(x) \pmod 2$$

In [ ]:
#Importación de librerías
import numpy as np
import hashlib
import time
import secrets
import math
from sympy import randprime

# Nota: usamos `secrets` para generar valores criptográficamente sensibles
# (el nonce r y la selección del secreto s), ya que `random` no es seguro para criptografía.
# `sympy.randprime` se usa para generar los primos p, q (uso matemático, no crítico en este contexto).
# La aritmética modular usa `pow(base, exp, mod)`, optimizada en C (exponenciación binaria).

In [ ]:

def generar_parametros(bits=512):
    """Genera las claves pública y privada para el protocolo Fiat-Shamir.

    Parámetros:
    - bits (int): Tamaño total deseado para el módulo N.

    Retorna:
    - dict: Diccionario con el módulo N, clave secreta s y clave pública v.
    """
    # 1. Generar dos primos grandes p y q de tamaño bits/2
    mitad_bits = bits // 2
    p = randprime(2 ** (mitad_bits - 1), 2**mitad_bits)
    q = randprime(2 ** (mitad_bits - 1), 2**mitad_bits)

    while p == q:
        q = randprime(2 ** (mitad_bits - 1), 2**mitad_bits)

    N = p * q

    # 2. Seleccionar clave secreta 's' tal que mcd(s, N) == 1
    while True:
        s = random.randint(2, N - 1)
        if math.gcd(s, N) == 1:
            break

    # 3. Calcular clave pública v = s^2 mod N
    v = pow(s, 2, N)

    return {"N": N, "s": s, "v": v}


def calcular_reto_fiat_shamir(compromiso):
    """Heurística de Fiat-Shamir: Reemplaza al Verificador calculando el reto e =

    H(x) mod 2 a partir del compromiso 'x'.
    """
    # Convertir el compromiso a bytes para la función hash SHA-256
    x_bytes = str(compromiso).encode("utf-8")
    hash_digest = hashlib.sha256(x_bytes).hexdigest()

    # Mapear el hash a un bit {0, 1}
    e = int(hash_digest, 16) % 2
    return e


params = generar_parametros(bits=512)
print("=== Parámetros Generados ===")
print(f"Módulo N ({params['N'].bit_length()} bits): {params['N']}")
print(f"Clave secreta (s): {params['s']}")
print(f"Clave pública (v): {params['v']}")


NameError: name 'random' is not defined

### 3.2 Prover: compromiso y respuesta


3.2. Compromiso del Prover (Commitment)En esta etapa, el Prover inicia el protocolo preparando una cegadora aleatoria que garantizará la propiedad de Cero Conocimiento (Zero-Knowledge):Selección del número aleatorio ($r$): El Prover elige de manera completamente aleatoria un valor efímero $r$ (llamado nonce) en el rango $1 < r < N$ tal que $\gcd(r, N) = 1$.Cálculo del compromiso ($x$): El Prover calcula la evidencia pública o compromiso mediante la operación modular:$$x \equiv r^2 \pmod N$$Seguridad del nonce: El valor de $r$ debe mantenerse estrictamente en secreto y debe ser regenerado para cada nuevo intento. Si un Prover reutiliza el mismo valor de $r$ para diferentes retos, un atacante podría recuperar la clave secreta $s$.

In [ ]:
def generar_compromiso(N):
    """Genera el número aleatorio secreto 'r' (nonce) y calcula el compromiso

    público 'x'.

    Parámetros:
    - N (int): El módulo del sistema generado en la sección 3.1.

    Retorna:
    - tuple: (r, x) donde 'r' es el valor aleatorio secreto y 'x' es el
    compromiso público.
    """
    while True:
        r = random.randint(2, N - 1)
        if math.gcd(r, N) == 1:
            break

    x = pow(r, 2, N)

    return r, x


r, x = generar_compromiso(params["N"])

print("=== Compromiso del Prover (3.2) ===")
print(f"Nonce secreto (r): {r}")
print(f"Compromiso público (x): {x}")

### 3.3 El reto vía hash (la transformación de Fiat-Shamir)


3.3. Obtención del Reto y Respuesta del Prover (Challenge & Response)En esta fase se realiza el cálculo del reto determinista y la construcción de la respuesta que se enviará al Verifier:Obtención del reto ($e$): A través de la heurística de Fiat-Shamir (definida en 3.1), el compromiso $x$ se pasa por la función Hash para fijar de forma no interactiva el valor del reto:$$e = H(x) \pmod 2 \quad \in \{0, 1\}$$Cálculo de la respuesta ($y$): El Prover responde mediante la fórmula:$$y \equiv r \cdot s^e \pmod N$$Análisis de los casos:Si $e = 0 \implies y \equiv r \pmod N$: El Prover demuestra que conoce la raíz cuadrada del compromiso $x$.Si $e = 1 \implies y \equiv r \cdot s \pmod N$: El Prover demuestra que conoce la clave secreta $s$ enmascarándola con la cegadora aleatoria $r$.Debido a que $r$ es un número totalmente aleatorio y único para esta sesión, ningún observador externo (ni siquiera el Verifier) puede despejar $s$ a partir de $y$.

In [ ]:
def generar_respuesta(r, s, e, N):
    """Calcula la respuesta del Prover: y = (r * (s^e)) mod N.

    Parámetros:
    - r (int): Nonce secreto del Prover.
    - s (int): Clave secreta del Prover.
    - e (int): Reto (0 o 1) devuelto por la función Hash.
    - N (int): Módulo del sistema.

    Retorna:
    - int: La respuesta 'y'.
    """
    y = (r * pow(s, e, N)) % N
    return y


e = calcular_reto_fiat_shamir(x)
y = generar_respuesta(r, params["s"], e, params["N"])

print("=== Generación de Reto y Respuesta (3.3) ===")
print(f"Reto derivado del Hash (e): {e}")
print(f"Respuesta construida (y):   {y}")

### 3.4 Verificación


3.4. Verificación de la Prueba por el Verificador (Verification)En la fase final, el Verificador recibe la tupla pública $(x, y)$ y realiza la validación sin necesidad de comunicarse interactivamente con el Prover:Reconstrucción del reto ($e$): El Verificador recalcula de manera idéntica el reto $e$ aplicando la función Hash al compromiso $x$:$$e = H(x) \pmod 2$$Comprobación de la identidad modular: El Verificador comprueba si se satisface la siguiente congruencia:$$y^2 \equiv x \cdot v^e \pmod N$$Demostración de corrección matemática:Sustituyendo la respuesta $y = r \cdot s^e \pmod N$ y la clave pública $v = s^2 \pmod N$:$$y^2 \equiv (r \cdot s^e)^2 \equiv r^2 \cdot (s^2)^e \equiv x \cdot v^e \pmod N$$Si $e = 0 \implies y^2 \equiv r^2 \equiv x \pmod N \quad $Si $e = 1 \implies y^2 \equiv r^2 \cdot s^2 \equiv x \cdot v \pmod N \quad $Si la igualdad se cumple, la prueba es aceptada como legítima.

In [ ]:
def verificar_prueba(x, y, v, N):
    """Verifica la validez de una prueba de Fiat-Shamir no interactiva.

    Parámetros:
    - x (int): Compromiso público del Prover.
    - y (int): Respuesta del Prover.
    - v (int): Clave pública del Prover (v = s^2 mod N).
    - N (int): Módulo del sistema.

    Retorna:
    - bool: True si la prueba es válida, False en caso contrario.
    """
    e_recalculado = calcular_reto_fiat_shamir(x)

    lhs = pow(y, 2, N)

    rhs = (x * pow(v, e_recalculado, N)) % N

    return lhs == rhs


es_valida = verificar_prueba(x, y, params["v"], params["N"])

print("=== Verificación de la Prueba (3.4) ===")
print(f"¿Prueba aceptada por el Verificador?: {es_valida}")

y_falsa = (y + 1) % params["N"]
es_valida_falsa = verificar_prueba(x, y_falsa, params["v"], params["N"])
print(f"¿Prueba alterada aceptada?: {es_valida_falsa}")

## 4. Ejemplo aplicado — Parte A: costo computacional

**Pregunta:** ¿cómo escala el costo de generar y verificar una prueba en función
del tamaño de N (256, 512, 1024, 2048 bits)?

Medimos por separado el tiempo de generar las claves (buscar los primos $p$, $q$ y calcular
$N$) y el tiempo del ciclo completo de la prueba (compromiso, reto, respuesta y verificación),
porque escalan de forma muy distinta.

**Resultado:** el keygen crece mucho más rápido que el ciclo de prueba. Al pasar de 256 a 2048
bits, el tiempo de keygen se multiplica por un factor de casi 444, mientras que el ciclo de
prueba solo se multiplica por 4. Esto confirma que el costo real de Fiat-Shamir para un Prover
recurrente es prácticamente despreciable frente al costo, pagado una sola vez, de generar las
claves.

In [ ]:
import time

tamanos_bits = [256, 512, 1024, 2048]
repeticiones = 5

tiempos_keygen = []
tiempos_prueba = []

for bits in tamanos_bits:
    t_keygen = []
    t_prueba = []
    for _ in range(repeticiones):
        # --- generación de parámetros ---
        t0 = time.perf_counter()
        params = generar_parametros(bits=bits)
        t_keygen.append(time.perf_counter() - t0)

        # --- ciclo completo de la prueba ---
        t0 = time.perf_counter()
        r, x = generar_compromiso(params["N"])
        e = calcular_reto_fiat_shamir(x)
        y = generar_respuesta(r, params["s"], e, params["N"])
        _ = verificar_prueba(x, y, params["v"], params["N"])
        t_prueba.append(time.perf_counter() - t0)

    tiempos_keygen.append(np.median(t_keygen))
    tiempos_prueba.append(np.median(t_prueba))
    print(f"{bits} bits -> keygen: {tiempos_keygen[-1]:.4f}s | prueba: {tiempos_prueba[-1]:.6f}s")

In [ ]:
from matplotlib.ticker import FixedLocator, FixedFormatter, NullFormatter

plt.close('all')
plt.figure(figsize=(7,5))
plt.loglog(tamanos_bits, tiempos_keygen, marker='o', color="#0072B2", label="Generación de claves")
plt.loglog(tamanos_bits, tiempos_prueba, marker='s', color="#E69F00", label="Ciclo de prueba")

ax = plt.gca()
ax.xaxis.set_major_locator(FixedLocator(tamanos_bits))
ax.xaxis.set_major_formatter(FixedFormatter([str(b) for b in tamanos_bits]))
ax.xaxis.set_minor_formatter(NullFormatter())

plt.xlabel(r"Tamaño de $N$ (bits)")
plt.ylabel(r"Tiempo (s)")
plt.title("Costo computacional de Fiat-Shamir vs. tamaño de N")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)

plt.savefig("costo_computacional.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Ejemplo aplicado — Parte B: análisis estadístico del hash (Monte Carlo)

**Pregunta:** ¿el hash se comporta como un generador de retos uniforme e impredecible
(modelo de oráculo aleatorio)? ¿Qué pasa si tuviera sesgos?

Para chequear si el hash se comporta como se supone que debe, lo generamos muchas veces y
verificamos si el resultado es parejo parejo. Con 100\,000 compromisos aleatorios, la
proporción de $e=0$ fue $0.4999$, prácticamente un 50/50. Además, la prueba chi-cuadrado
confirma que esto no es casualidad ($\chi^2 = 0.0090$, $p = 0.9244$): el hash no muestra ningún
sesgo.

En conjunto con el histograma del primer byte del hash (que tampoco muestra picos ni huecos
apreciables), estos resultados son evidencia empírica de que SHA-256 se comporta, en este
experimento, de forma consistente con el modelo de oráculo aleatorio.

In [ ]:
n_simulaciones = 100_000
retos_bit = []

for _ in range(n_simulaciones):
    compromiso = random.randint(0, 2**256)
    e = calcular_reto_fiat_shamir(compromiso)
    retos_bit.append(e)

retos_bit = np.array(retos_bit)
prop_ceros = np.mean(retos_bit == 0)
print(f"Proporción de e=0: {prop_ceros:.4f}  (esperado: 0.5)")


In [ ]:
from scipy.stats import chisquare

obs = np.array([np.sum(retos_bit==0), np.sum(retos_bit==1)])
esperado = np.array([n_simulaciones/2, n_simulaciones/2])
chi2, p_valor = chisquare(obs, esperado)
print(f"chi2 = {chi2:.4f}, p-valor = {p_valor:.4f}")


In [ ]:
n_simulaciones = 20_000  # suficiente para el histograma de bytes
primer_byte = []

for _ in range(n_simulaciones):
    compromiso = random.randint(0, 2**256)
    x_bytes = str(compromiso).encode("utf-8")
    digest = hashlib.sha256(x_bytes).digest()
    primer_byte.append(digest[0])  # primer byte del hash, 0-255

primer_byte = np.array(primer_byte)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,4))

axs[0].bar([0,1], [np.sum(retos_bit==0), np.sum(retos_bit==1)],
           color=["#0072B2", "#E69F00"])
axs[0].axhline(n_simulaciones/2, color="#000000", linestyle="--", label="esperado")
axs[0].set_xticks([0,1]); axs[0].set_title(r"Distribución del reto $e$")
axs[0].set_xlabel(r"$e$"); axs[0].set_ylabel("Frecuencia"); axs[0].legend()

axs[1].hist(primer_byte, bins=32, color="#009E73", edgecolor="black")
axs[1].set_title(r"Distribución del primer byte de $H(x)$")
axs[1].set_xlabel("Valor del byte (0-255)"); axs[1].set_ylabel("Frecuencia")

plt.tight_layout()
fig.savefig("distribucion_hash.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Conclusiones

_(síntesis breve: qué muestran las partes A y B, y cómo conectan con la seguridad
y eficiencia real de Fiat-Shamir)_



Por el lado del costo, lo que vimos es que el protocolo en sí: compromiso, reto,
respuesta y verificación es eficiente incluso con $N$ de 2048 bits, el tiempo
sigue siendo del orden de microsegundos. Lo caro es generar las claves al principio, porque ahí
sí hay que buscar primos grandes y esto escala rápido. Pero como las claves se generan una sola
vez y se reutilizan para muchas pruebas, ese costo se diluye en el tiempo: se paga caro al inicio,
pero después cada prueba es instantánea.

Por el lado del hash, la simulación Monte Carlo nos dio justo lo que esperaría el modelo
teórico: el reto sale 50/50 sin ningún sesgo detectable, y la prueba chi-cuadrado lo respalda.
Toda la garantía de \emph{soundness} del protocolo (que un
impostor solo tenga $1/2$ de probabilidad de colar una prueba falsa por ronda) depende de que el
reto sea realmente impredecible. Aunque esto no es una prueba
matemática de que SHA-256 sea un oráculo aleatorio perfecto, sí es evidencia empírica a favor.

## 7. Opinión del grupo


- **David Delgado Ortiz:** _(pendiente)_
- **Miguel Angel Fonseca Aldana:** En Criptografía solo había visto la parte teórica y no lo había puesto en práctica hasta este trabajo. Me pareció un buen ejercicio poder aplicarlo en Programación Científica, sobre todo viendo lo del método de Montecarlo.
- **Jose Daniel Moreno Ceballos:** _(pendiente)_
- **Juan Diego Ospina Ocampo:** Es el problema de seguridad informatíca clásico, también se relaciona al análisis numérico, siento que es perfecto para la evaluación, me gusta el enfoque de la seguridad, para esta entrega pues es un ejemplo perfecto, verificado y que nos dejó experimentar como se comporta con diferentes valores.
- **Haison Urrutia Manyoma:** _(pendiente)_
